In [ ]:
import torch

from dataset import *
from model import ResNet18
from unlearn import *
from utils import *
from torch.utils.data import DataLoader

In [ ]:
train_ds = CustomCIFAR100(root='.', train=True,download=True, transform=transform_train)
valid_ds = CustomCIFAR100(root='.', train=False,download=True, transform=transform_train)

batch_size = 256
train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=32, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size, num_workers=32, pin_memory=True)

In [ ]:
num_classes = 100
classwise_train = {}
for i in range(num_classes):
    classwise_train[i] = []

for img, label, clabel in train_ds:
    classwise_train[label].append((img, label, clabel))
    
classwise_test = {}
for i in range(num_classes):
    classwise_test[i] = []

for img, label, clabel in valid_ds:
    classwise_test[label].append((img, label, clabel))

In [ ]:
# train the model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ResNet18(num_classes = 20, pretrained = True).to(device)
epochs = 5
history = fit_one_cycle(epochs, model, train_dl, valid_dl, device = device)

torch.save(model.state_dict(), "ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt")

In [ ]:
# load the trained model
device = 'cuda'
model = ResNet18(num_classes = 20, pretrained = True).to(device)
model.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt", map_location='cuda'))

# Forgetting Rocket
The Rocket is class 69 in CIFAR100 and belongs to Super Class 19 (Vehicles) in CIFAR Super 20.

In [ ]:
# Getting the forget and retain validation data
forget_valid = []
forget_classes = [69]
for cls in range(num_classes):
    if cls in forget_classes:
        for img, label, clabel in classwise_test[cls]:
            forget_valid.append((img, label, clabel))

retain_valid = []
for cls in range(num_classes):
    if cls not in forget_classes:
        for img, label, clabel in classwise_test[cls]:
            retain_valid.append((img, label, clabel))
            
forget_train = []
for cls in range(num_classes):
    if cls in forget_classes:
        for img, label, clabel in classwise_train[cls]:
            forget_train.append((img, label, clabel))

retain_train = []
for cls in range(num_classes):
    if cls not in forget_classes:
        for img, label, clabel in classwise_train[cls]:
            retain_train.append((img, label, clabel))

forget_valid_dl = DataLoader(forget_valid, batch_size, num_workers=32, pin_memory=True)

retain_valid_dl = DataLoader(retain_valid, batch_size, num_workers=32, pin_memory=True)

forget_train_dl = DataLoader(forget_train, batch_size, num_workers=32, pin_memory=True)
retain_train_dl = DataLoader(retain_train, batch_size, num_workers=32, pin_memory=True, shuffle = True)
import random
retain_train_subset = random.sample(retain_train, int(0.3*len(retain_train)))
retain_train_subset_dl = DataLoader(retain_train_subset, batch_size, num_workers=32, pin_memory=True, shuffle = True)

In [ ]:
print("Forget Performance: {}".format(evaluate(model, forget_valid_dl, device)))
print("Retain Performance: {}".format(evaluate(model, retain_valid_dl, device)))

## Retrain the model from Scratch (Optional)
Create Retrained Model (Gold model). This is the model trained from scratch without forget data.

In [ ]:
device = 'cuda'
gold_model = ResNet18(num_classes = 20, pretrained = True).to(device)
epochs = 5
history = fit_one_cycle(epochs, gold_model, retain_train_dl, retain_valid_dl, device = device)
torch.save(gold_model.state_dict(), "ResNET18_CIFAR100Super20_Pretrained_Gold_Class69_5_Epochs.pt")

In [ ]:
device = 'cuda'
gold_model = ResNet18(num_classes = 20, pretrained = True).to(device)
gold_model.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_Gold_Class69_5_Epochs.pt", map_location=device))

In [ ]:
# evaluate gold model on forget set
evaluate(gold_model, forget_valid_dl, device)

In [ ]:
# evaluate gold model on retain set
evaluate(gold_model, retain_valid_dl, device)

## Unlearning using Amnesiac unlearning

In [ ]:
unlearninglabels = list(range(20))
unlearninglabels.remove(19)
unlearning_train_set = []
for cls in range(num_classes):
    if cls in forget_classes:
        for img, label, clabel in classwise_train[cls]:
            unlearning_train_set.append((img, label, random.choice(unlearninglabels)))



for cls in range(num_classes):
    if cls not in forget_classes:
        for img, label, clabel in classwise_train[cls]:
            unlearning_train_set.append((img, label, clabel))

In [ ]:
unlearning_train_set_dl = DataLoader(unlearning_train_set, batch_size, num_workers = 32, pin_memory = True, shuffle = True)

In [ ]:
device = 'cuda'
student_model = ResNet18(num_classes = 20, pretrained = True).to(device)
student_model.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt", map_location = 'cuda'))
epochs = 3

history = fit_one_unlearning_cycle(epochs, student_model, unlearning_train_set_dl, retain_valid_dl, device = device, lr = 0.0001)

In [ ]:
print("Forget Performance: {}".format(evaluate(student_model, forget_valid_dl, device)))
print("Retain Performance: {}".format(evaluate(student_model, retain_valid_dl, device)))

## UnLearning using bad teacher

In [ ]:
device = 'cuda'
unlearning_teacher = ResNet18(num_classes = 20, pretrained = False).to(device).eval()
student_model = ResNet18(num_classes = 20, pretrained = False).to(device)
student_model.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt", map_location = device))
model = model.eval()

KL_temperature = 1

optimizer = torch.optim.Adam(student_model.parameters(), lr = 0.0001)

blindspot_unlearner(model = student_model, unlearning_teacher = unlearning_teacher, full_trained_teacher = model, 
          retain_data = retain_train_subset, forget_data = forget_train, epochs = 1, optimizer = optimizer, lr = 0.0001, 
          batch_size = 256, num_workers = 32, device = device, KL_temperature = KL_temperature)

In [ ]:
print("Forget Performance: {}".format(evaluate(student_model, forget_valid_dl, device)))
print("Retain Performance: {}".format(evaluate(student_model, retain_valid_dl, device)))

# 统一评估：准确率 + MIA + ZRF 指标对比
下面的 cell 在同一套数据划分下，依次评估 **原始模型 / Amnesiac / Bad Teacher / Bad Teacher + Retain(本文改进)** 四个模型，报告：
- **Forget Acc**：遗忘类（火箭）验证准确率，越低越好；
- **Retain Acc**：保留类验证准确率，越高越好；
- **MIA**：成员推断攻击将 forget 样本判为"训练成员"的比例，越低越好；
- **ZRF**：遗忘集上与随机初始化教师的行为接近度，越高越好。

结果保存为 `unlearning_results.csv`，可直接填入实验报告表格。
> 运行前请确保已执行上文的模型训练、forget/retain 数据划分、以及 Amnesiac 的 `unlearning_train_set_dl` 构造 cell。

In [ ]:
import copy
import pandas as pd
from metrics import get_membership_attack_prob, compute_zrf

CKPT = "ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt"


def load_base():
    m = ResNet18(num_classes=20, pretrained=False).to(device)
    m.load_state_dict(torch.load(CKPT, map_location=device))
    return m


# A single randomly-initialised ("incompetent") teacher shared by
# the bad-teacher methods and the ZRF metric.
incompetent_teacher = ResNet18(num_classes=20, pretrained=False).to(device).eval()


def full_eval(name, m):
    """Accuracy on forget/retain + MIA + ZRF for one model."""
    m.eval()
    forget_acc = evaluate(m, forget_valid_dl, device)["Acc"]
    retain_acc = evaluate(m, retain_valid_dl, device)["Acc"]
    # Attacker: members = retain-train subset, non-members = retain-valid (unseen),
    # query = forget-train (were members before unlearning).
    mia = get_membership_attack_prob(retain_train_subset_dl, forget_train_dl, retain_valid_dl, m, device)
    zrf = compute_zrf(m, incompetent_teacher, forget_valid_dl, device)
    print(f"[{name:26s}] Forget={forget_acc:6.2f}  Retain={retain_acc:6.2f}  MIA={mia:.3f}  ZRF={zrf:.3f}")
    return dict(Method=name, Forget_Acc=round(forget_acc, 2), Retain_Acc=round(retain_acc, 2),
                MIA=round(mia, 3), ZRF=round(zrf, 3))


rows = []

# (0) Original model, before any unlearning.
rows.append(full_eval("Original", load_base()))

# (1) Amnesiac unlearning (random relabelling + fine-tune).
amnesiac = load_base()
fit_one_unlearning_cycle(3, amnesiac, unlearning_train_set_dl, retain_valid_dl, lr=0.0001, device=device)
rows.append(full_eval("Amnesiac", amnesiac))

# (2) Bad Teacher unlearning (blindspot distillation).
badteacher = load_base()
blindspot_unlearner(
    model=badteacher, unlearning_teacher=incompetent_teacher, full_trained_teacher=load_base(),
    retain_data=retain_train_subset, forget_data=forget_train, epochs=1, lr=0.0001,
    batch_size=256, num_workers=0, device=device, KL_temperature=1,
)
rows.append(full_eval("Bad Teacher", badteacher))

# (3) Retain-Aware Bad Teacher (our improvement).
ra = load_base()
blindspot_unlearner_retain_aware(
    model=ra, unlearning_teacher=incompetent_teacher, full_trained_teacher=load_base(),
    retain_data=retain_train_subset, forget_data=forget_train, epochs=1, lr=0.0001,
    batch_size=256, num_workers=0, device=device, KL_temperature=1, retain_ce_weight=1.0,
)
rows.append(full_eval("Bad Teacher + Retain (Ours)", ra))

summary = pd.DataFrame(rows)
summary.to_csv("unlearning_results.csv", index=False)
summary

## 结果可视化
准确率对比（Forget 越低越好、Retain 越高越好）与隐私指标对比（MIA 越低越好、ZRF 越高越好）。图片会保存为 `fig_accuracy_comparison.png` 与 `fig_privacy_metrics.png`，可直接放进报告。

In [ ]:
import matplotlib.pyplot as plt

methods = summary["Method"].tolist()
x = range(len(methods))
width = 0.35

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar([i - width / 2 for i in x], summary["Forget_Acc"], width, label="Forget Acc (lower=better)", color="#d1495b")
ax1.bar([i + width / 2 for i in x], summary["Retain_Acc"], width, label="Retain Acc (higher=better)", color="#2e86ab")
ax1.set_ylabel("Accuracy (%)")
ax1.set_xticks(list(x))
ax1.set_xticklabels(methods, rotation=20, ha="right")
ax1.set_title("Forget vs Retain accuracy across unlearning methods")
ax1.legend()
plt.tight_layout()
plt.savefig("fig_accuracy_comparison.png", dpi=150)
plt.show()

# Privacy metrics (MIA / ZRF)
fig, ax2 = plt.subplots(figsize=(9, 5))
ax2.bar([i - width / 2 for i in x], summary["MIA"], width, label="MIA (lower=better)", color="#edae49")
ax2.bar([i + width / 2 for i in x], summary["ZRF"], width, label="ZRF (higher=better)", color="#66a182")
ax2.set_ylabel("Score")
ax2.set_xticks(list(x))
ax2.set_xticklabels(methods, rotation=20, ha="right")
ax2.set_title("Privacy / forgetting metrics across unlearning methods")
ax2.legend()
plt.tight_layout()
plt.savefig("fig_privacy_metrics.png", dpi=150)
plt.show()